# AI Video Generation Agent Pipeline (Production-Grade)
This notebook implements a complete, production-ready, and end-to-end AI Video Generation Agent. It translates a narration script, image assets, and metadata into a synchronized, cinematic video clip.

### 🌟 Features:
- **TTS Narration Engine** with Kokoro-TTS and robust gTTS fallback.
- **Timing Synchronization**: Uses audio duration as the single source of truth, mathematically eliminating drift or timeline overlaps.
- **Structured Timeline Planning**: Employs Groq LLM API to dynamically plan image sequences, transitions, and cinematic effects.
- **Rigorous Validation & Normalization Layer**: Prevents invalid schemas, missing assets, unauthorized effects/transitions, or timing mismatch.
- **Cinematic Rendering & Direct Muxing**: Combines MoviePy and high-speed FFmpeg muxing to create high-performance MP4 files (H.264/AAC).
- **Gradio Interactive UI & FastAPI Endpoints** for out-of-the-box local testing and production deployment (Hugging Face Spaces).



## 1. Install Dependencies
We install the necessary libraries for media processing, schemas, model execution, and UI creation.


In [ ]:
# Install required packages
import sys
import subprocess

def install_dependencies():
    print("Installing production dependencies...")
    dependencies = [
        "moviepy>=1.0.3",
        "gTTS",
        "pillow",
        "numpy",
        "groq",
        "pydantic>=2.0.0",
        "fastapi",
        "uvicorn",
        "gradio",
        "soundfile",
        "kokoro"
    ]
    # Execute pip install
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + dependencies)
    print("Dependencies installed successfully!")

install_dependencies()


## 2. Imports
Import core modules for logic, schema building, LLM connection, and audio/video rendering.


In [ ]:
import os
import sys
import json
import time
import uuid
import shutil
import random
import logging
import subprocess
from typing import List, Dict, Optional, Tuple, Any
from pathlib import Path

# Media & Image Processing
import numpy as np
from PIL import Image, ImageDraw, ImageFont

# Structured Schema Definition
from pydantic import BaseModel, Field, ValidationError

# LLM Client
from groq import Groq

# TTS Fallback
from gtts import gTTS

# Web Services & Interface
import gradio as gr
from fastapi import FastAPI, HTTPException, Form
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import HTMLResponse, FileResponse

# Setup logger
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("AIVideoAgent")

# Dynamic MoviePy Import (supports MoviePy 1.x & 2.x)
try:
    # MoviePy v2.x
    from moviepy.video.io.ImageClip import ImageClip
    from moviepy.video.compositing.concatenate import concatenate_videoclips
    from moviepy.audio.io.AudioFileClip import AudioFileClip
    from moviepy.video.io.VideoFileClip import VideoFileClip
    import moviepy.video.fx as vfx
    logger.info("Loaded MoviePy v2.x dependencies.")
except ImportError:
    # MoviePy v1.x fallback
    try:
        from moviepy.editor import ImageClip, concatenate_videoclips, AudioFileClip, VideoFileClip
        import moviepy.video.fx.all as vfx
        logger.info("Loaded MoviePy v1.x dependencies.")
    except ImportError as e:
        logger.error(f"MoviePy imports failed: {e}. Run Section 1 dependency install first.")


## 3. Environment Variables
Securely load `GROQ_API_KEY` from either Google Colab Userdata Secrets or local shell environment variables.


In [ ]:
# Load API Key
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    try:
        from google.colab import userdata
        GROQ_API_KEY = userdata.get("GROQ_API_KEY")
        logger.info("GROQ_API_KEY successfully loaded from Colab Secret Keys.")
    except ImportError:
        logger.warning("Colab userdata module unavailable. Check environment variables.")

if not GROQ_API_KEY:
    logger.warning("GROQ_API_KEY is not defined. LLM timeline generator will fall back to a rules-based generator.")
else:
    logger.info("GROQ_API_KEY is loaded and ready.")


## 4. Upload Assets
Utility functions to manage user assets. To make this notebook fully executable out-of-the-box, we generate stunning premium gradient images if no files are uploaded manually.


In [ ]:
# Ensure target directories exist
os.makedirs("assets", exist_ok=True)
os.makedirs("output", exist_ok=True)

def get_premium_font(size: int):
    # Search common system font paths for clean rendering
    font_paths = [
        "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
        "C:\\Windows\\Fonts\\arialbd.ttf",
        "/System/Library/Fonts/Helvetica.ttc"
    ]
    for path in font_paths:
        if os.path.exists(path):
            try:
                return ImageFont.truetype(path, size)
            except Exception:
                continue
    return ImageFont.load_default()

def create_gradient_image(filename: str, color1: Tuple[int, int, int], color2: Tuple[int, int, int], text: str, width: int = 1920, height: int = 1080):
    """
    Generates a premium visual asset (1080p full HD gradient) with styled typography
    so the pipeline is fully executable out-of-the-box without requiring manual image uploads.
    """
    image = Image.new("RGB", (width, height))
    draw = ImageDraw.Draw(image)
    
    # Draw horizontal gradient
    for y in range(height):
        ratio = y / height
        r = int(color1[0] * (1 - ratio) + color2[0] * ratio)
        g = int(color1[1] * (1 - ratio) + color2[1] * ratio)
        b = int(color1[2] * (1 - ratio) + color2[2] * ratio)
        draw.line([(0, y), (width, y)], fill=(r, g, b))
        
    # Draw thin elegant inner border
    draw.rectangle([40, 40, width - 40, height - 40], outline=(255, 255, 255, 30), width=2)
    
    # Draw abstract background dots
    random.seed(text)
    for _ in range(40):
        x = random.randint(100, width - 100)
        y = random.randint(100, height - 100)
        radius = random.randint(3, 8)
        draw.ellipse([x-radius, y-radius, x+radius, y+radius], fill=(255, 255, 255, 25))

    # Add text overlay
    font = get_premium_font(55)
    
    # Draw dark shadow behind text
    draw.text((width // 2 - 300 + 4, height // 2 - 50 + 4), text, fill=(0, 0, 0, 150), font=font)
    # Draw primary text
    draw.text((width // 2 - 300, height // 2 - 50), text, fill=(255, 255, 255), font=font)
    
    path = os.path.join("assets", filename)
    image.save(path, "JPEG", quality=95)
    logger.info(f"Automatically generated placeholder asset: {path}")
    return path

# Autogenerate default premium placeholder graphics
PRESET_IMAGES = [
    {"id": "img1", "filename": "img1.jpg", "color1": (26, 41, 128), "color2": (38, 208, 206), "text": "THE POWER OF AI SYSTEMS", "desc": "Visualization of neural networks"},
    {"id": "img2", "filename": "img2.jpg", "color1": (74, 0, 224), "color2": (142, 45, 226), "text": "CYBERNETIC METROPOLIS", "desc": "Cyberpunk high-tech city"},
    {"id": "img3", "filename": "img3.jpg", "color1": (252, 74, 26), "color2": (247, 177, 114), "text": "CREATIVE COGNITION ENGINE", "desc": "Conceptual abstract thought"}
]

for img in PRESET_IMAGES:
    full_path = os.path.join("assets", img["filename"])
    if not os.path.exists(full_path):
        create_gradient_image(img["filename"], img["color1"], img["color2"], img["text"])


## 5. Pydantic Models
Strictly defined typing and schema constraints for inputs, individual visual timeline nodes, and the aggregate system layout.


In [ ]:
class ImageMeta(BaseModel):
    id: str = Field(..., description="Unique alphanumeric identifier for the asset (e.g. img1)")
    path: str = Field(..., description="Local path to the image file")
    description: Optional[str] = Field(None, description="Explaining what the image visualizes to guide the LLM planning")

class TimelineItem(BaseModel):
    image_id: str = Field(..., description="Target image reference")
    duration: float = Field(..., description="Display length in seconds")
    effect: str = Field("fade", description="Allowed values: zoom_in, fade, none")
    transition: str = Field("fade", description="Allowed values: fade, crossfade, none")

class Timeline(BaseModel):
    items: List[TimelineItem] = Field(..., description="Sequential visual sequence elements")
    total_duration: float = Field(..., description="The synchronized timeline total length")


## 6. TTS Engine
Converts the text narration script into an audio file. The engine attempts to load the high-fidelity `Kokoro-TTS` framework, and falls back to standard `gTTS` to guarantee error-free execution.


In [ ]:
class TTSEngine:
    def __init__(self, output_dir="output", prefer_kokoro=True):
        self.output_dir = output_dir
        self.prefer_kokoro = prefer_kokoro
        os.makedirs(output_dir, exist_ok=True)
        
    def generate(self, script: str) -> Tuple[str, float]:
        """
        Generates narrative audio file for a given text script.
        Returns:
            Tuple[str, float]: (audio_file_path, exact_duration_in_seconds)
        """
        if not script or not script.strip():
            raise ValueError("Narration script is empty.")
            
        filename = f"tts_{uuid.uuid4().hex[:8]}.mp3"
        audio_path = os.path.join(self.output_dir, filename)
        
        # 1. Try Kokoro-TTS if active
        if self.prefer_kokoro:
            try:
                logger.info("Attempting high-fidelity Kokoro-TTS generation...")
                from kokoro import KPipeline
                import soundfile as sf
                
                pipeline = KPipeline(lang_code='a') # American English
                generator = pipeline(script, voice='af_heart', speed=1.0)
                
                audio_chunks = []
                for _, _, audio in generator:
                    if audio is not None:
                        audio_chunks.append(audio)
                        
                if audio_chunks:
                    full_audio = np.concatenate(audio_chunks)
                    wav_filename = f"tts_{uuid.uuid4().hex[:8]}.wav"
                    audio_path = os.path.join(self.output_dir, wav_filename)
                    
                    sf.write(audio_path, full_audio, 24000) # Kokoro operates at 24kHz
                    duration = len(full_audio) / 24000.0
                    logger.info(f"Kokoro-TTS generation successful. Duration: {duration:.2f}s")
                    return audio_path, duration
                else:
                    raise Exception("Kokoro engine returned empty audio tracks.")
            except Exception as e:
                logger.warning(f"Kokoro-TTS initialization failed ({e}). Falling back to robust gTTS.")
                
        # 2. Robust gTTS Fallback
        try:
            logger.info("Running gTTS voice synthesizer...")
            tts = gTTS(text=script, lang='en', slow=False)
            tts.save(audio_path)
            
            # Extract duration safely using MoviePy
            try:
                clip = AudioFileClip(audio_path)
                duration = clip.duration
                clip.close()
            except Exception as clip_err:
                logger.warning(f"MoviePy failed to read audio duration: {clip_err}. Using word estimation fallback.")
                words = len(script.split())
                duration = max(2.5, words / 2.5) # Estimate 150 words per minute
                
            logger.info(f"gTTS audio generated. Duration: {duration:.2f}s")
            return audio_path, duration
        except Exception as e:
            logger.critical(f"All TTS synthesis engines failed: {e}")
            raise RuntimeError(f"TTS Engine Failure: {e}")


## 7. Groq LLM Setup
Wraps the official Groq client with fallback error handling, running in rules-based simulation mode if no API key is specified.


In [ ]:
class GroqLLM:
    def __init__(self, api_key: Optional[str] = None):
        self.api_key = api_key or GROQ_API_KEY
        self.client = None
        if self.api_key:
            try:
                self.client = Groq(api_key=self.api_key)
                logger.info("Groq API client successfully initialized.")
            except Exception as e:
                logger.error(f"Failed to initialize Groq client: {e}")
        else:
            logger.warning("Groq API Key not detected. The generator will run using local rule-based sequencing.")
            
    def get_client(self) -> Optional[Groq]:
        return self.client


## 8. Prompt Engineering
Systematically structures constraints, metadata details, audio duration limits, and visual instructions into a highly detailed prompt targeting perfect JSON outputs from the LLM.


In [ ]:
def build_cinematic_prompt(script: str, images: List[ImageMeta], audio_duration: float) -> str:
    image_list_str = "\n".join([f"- ID: '{img.id}' | Description: {img.description or 'Visual Asset'}" for img in images])
    allowed_ids = [img.id for img in images]
    
    prompt = f"""
ROLE: You are an expert AI Video Editor and Director. Your task is to plan a highly cinematic, perfectly-synchronized video timeline.

INPUT NARRATION SCRIPT:
"{script}"

TARGET AUDIO DURATION:
{audio_duration:.2f} seconds. (CRITICAL: The sum of the durations of all visual items MUST match this EXACT duration perfectly).

AVAILABLE IMAGE ASSETS (ONLY use these IDs, do NOT hallucinate other IDs):
{image_list_str}

RULES & CONSTRAINTS:
1. You must assign each visual segment a specific image_id from the allowed list: {allowed_ids}.
2. Ensure every single image ID is used at least once if possible, but keep sequence cinematic. Do not hallucinate image IDs that are not in the list.
3. Max duration per image: 8.0 seconds. Min duration per image: 1.5 seconds.
4. Allowed Effects: "zoom_in", "fade", "none".
5. Allowed Transitions: "fade", "crossfade", "none".
6. Audio Sync: The 'duration' field of each item is the time in seconds that this image stays on screen. The sum of all item durations MUST sum up to exactly {audio_duration:.2f}.
7. Return ONLY a valid JSON string. Do not include markdown codeblocks (no ```json ... ```), no explanations, no trailing commas, and no intro/outro text.

Strict Output JSON Schema Example:
{{
  "items": [
    {{
      "image_id": "{allowed_ids[0]}",
      "duration": 3.5,
      "effect": "zoom_in",
      "transition": "fade"
    }},
    {{
      "image_id": "{allowed_ids[1] if len(allowed_ids) > 1 else allowed_ids[0]}",
      "duration": {audio_duration - 3.5:.2f},
      "effect": "fade",
      "transition": "fade"
    }}
  ],
  "total_duration": {audio_duration:.2f}
}}
"""
    return prompt


## 9. Timeline Generator
Queries the Groq API using model `llama-3.3-70b-versatile`. Includes robust JSON decoding fallbacks and a deterministic rule-based generator that handles rate-limits or invalid api keys.


In [ ]:
class TimelineGenerator:
    def __init__(self, groq_wrapper: GroqLLM, model_name: str = "llama-3.3-70b-versatile"):
        self.groq_wrapper = groq_wrapper
        self.model_name = model_name

    def generate_timeline(self, script: str, images: List[ImageMeta], audio_duration: float) -> Dict[str, Any]:
        """
        Queries Groq LLM API to build the cinematic visual timeline, with automatic heuristics fallback.
        """
        client = self.groq_wrapper.get_client()
        prompt = build_cinematic_prompt(script, images, audio_duration)
        
        if client:
            try:
                logger.info(f"Querying Groq LLM model: {self.model_name}...")
                response = client.chat.completions.create(
                    messages=[
                        {"role": "system", "content": "You are a professional video timeline compiler. You output ONLY raw JSON conformant to the requested schema."},
                        {"role": "user", "content": prompt}
                    ],
                    model=self.model_name,
                    temperature=0.2,
                    max_tokens=1000
                )
                
                raw_response = response.choices[0].message.content.strip()
                
                # Strip markdown syntax wraps if outputted
                if raw_response.startswith("```"):
                    raw_response = raw_response.strip("`").replace("json", "", 1).strip()
                
                timeline_data = json.loads(raw_response)
                logger.info("Successfully fetched and parsed LLM cinematic timeline plan.")
                return timeline_data
                
            except json.JSONDecodeError as jde:
                logger.warning(f"Groq API returned invalid JSON. Attempting string parsing extraction...")
                try:
                    start_idx = raw_response.find("{")
                    end_idx = raw_response.rfind("}")
                    if start_idx != -1 and end_idx != -1:
                        return json.loads(raw_response[start_idx:end_idx+1])
                except Exception:
                    pass
            except Exception as e:
                logger.error(f"Groq API call encountered a failure: {e}. Reverting to local rule-based scheduler.")
                
        # Heuristic Deterministic Planner Fallback
        logger.info("Triggering deterministic rule-based visual scheduler...")
        num_images = len(images)
        if num_images == 0:
            raise ValueError("Cannot schedule a timeline without image assets.")
            
        avg_dur = audio_duration / num_images
        items = []
        effects = ["zoom_in", "fade", "none"]
        transitions = ["fade", "none"]
        
        for i, img in enumerate(images):
            items.append({
                "image_id": img.id,
                "duration": round(avg_dur, 2),
                "effect": effects[i % len(effects)],
                "transition": transitions[i % len(transitions)]
            })
            
        return {
            "items": items,
            "total_duration": audio_duration
        }


## 10. Validation Layer
Rigorous schema and logical value check verifying every single field in the timeline structure, filtering unknown image references, bad values, or disallowed visual options.


In [ ]:
class TimelineValidator:
    @staticmethod
    def validate(timeline_data: Dict[str, Any], available_images: List[ImageMeta], expected_duration: float) -> Timeline:
        """
        Performs rigorous structural validation and mapping. Raises ValueError upon breach.
        """
        logger.info("Executing rigorous schema validation on timeline data...")
        
        if "items" not in timeline_data or not isinstance(timeline_data["items"], list):
            raise ValueError("Timeline structure is invalid: 'items' array missing.")
            
        items = []
        valid_ids = {img.id for img in available_images}
        
        for i, raw_item in enumerate(timeline_data["items"]):
            if "image_id" not in raw_item or "duration" not in raw_item:
                raise ValueError(f"Timeline item at index {i} is missing 'image_id' or 'duration'.")
                
            img_id = raw_item["image_id"]
            if img_id not in valid_ids:
                raise ValueError(f"Timeline item at index {i} references unauthorized image ID: '{img_id}'")
                
            try:
                dur = float(raw_item["duration"])
            except ValueError:
                raise ValueError(f"Timeline item at index {i} duration '{raw_item['duration']}' is not numeric.")
                
            if dur <= 0:
                raise ValueError(f"Timeline item at index {i} has an invalid non-positive duration: {dur}")
                
            # Safely clamp visual choices to valid options
            effect = raw_item.get("effect", "none").lower()
            transition = raw_item.get("transition", "none").lower()
            
            if effect not in {"zoom_in", "fade", "none"}:
                effect = "none"
            if transition not in {"fade", "crossfade", "none"}:
                transition = "none"
                
            items.append(TimelineItem(
                image_id=img_id,
                duration=dur,
                effect=effect,
                transition=transition
            ))
            
        validated_timeline = Timeline(items=items, total_duration=sum(item.duration for item in items))
        logger.info("Timeline successfully passed all system validation assertions.")
        return validated_timeline


## 11. Timeline Normalization
Performs scaling calculations to stretch/shrink the timeline segments proportionally, matching the exact TTS audio length down to the millisecond, avoiding timing gaps, drifts, or hangs.


In [ ]:
class TimelineNormalizer:
    @staticmethod
    def normalize(timeline: Timeline, target_duration: float) -> Timeline:
        """
        Scales all timeline segments proportionally to match target_duration EXACTLY,
        mitigating floating point roundoff errors down to the millisecond.
        """
        logger.info(f"Scaling visual timeline to target duration: {target_duration:.4f}s...")
        
        current_sum = sum(item.duration for item in timeline.items)
        if current_sum <= 0:
            raise ValueError("Aggregate timeline duration is zero. Unable to scale.")
            
        scale_ratio = target_duration / current_sum
        
        for item in timeline.items:
            item.duration = round(item.duration * scale_ratio, 4)
            
        # Eliminate final float rounding differences in the last element
        new_sum = sum(item.duration for item in timeline.items)
        diff = target_duration - new_sum
        
        if abs(diff) > 0.0001:
            timeline.items[-1].duration = round(timeline.items[-1].duration + diff, 4)
            
        timeline.total_duration = target_duration
        logger.info(f"Timeline scaled successfully. Cumulative length: {sum(item.duration for item in timeline.items):.4f}s")
        return timeline


## 12. Effects Engine
Encapsulates modern visual transformation effects (e.g. Ken Burns zoom-in, cross-dissolving) utilizing a modular architecture wrapping MoviePy vfx transforms.


In [ ]:
class EffectsEngine:
    @staticmethod
    def apply_zoom_in(clip, duration: float, max_zoom: float = 1.12):
        """
        Applies a smooth cinematic zoom-in effect over the clip duration.
        """
        try:
            # Resize callback function
            def zoom_func(t):
                factor = 1.0 + (max_zoom - 1.0) * (t / duration)
                return factor
                
            if hasattr(clip, "resize"):
                return clip.resize(zoom_func)
            elif hasattr(clip, "with_effects"):
                return clip.with_effects([vfx.Resize(zoom_func)])
            else:
                return clip.fx(vfx.resize, zoom_func)
        except Exception as e:
            logger.error(f"Zoom effect failed: {e}. Skipping transform.")
            return clip

    @staticmethod
    def apply_fade(clip, duration: float):
        """
        Applies clean, atmospheric fade in and out bounds.
        """
        try:
            fade_dur = min(0.6, duration / 3.0)
            if hasattr(clip, "fadein") and hasattr(clip, "fadeout"):
                return clip.fadein(fade_dur).fadeout(fade_dur)
            elif hasattr(clip, "with_effects"):
                return clip.with_effects([vfx.FadeIn(fade_dur), vfx.FadeOut(fade_dur)])
            else:
                return clip.fx(vfx.fadein, fade_dur).fx(vfx.fadeout, fade_dur)
        except Exception as e:
            logger.error(f"Fade effect failed: {e}. Skipping transform.")
            return clip


## 13. Transition Engine
Handles transition overlays between visual elements, implementing reliable in-clip fades to protect against track overlapping rendering crashes.


In [ ]:
class TransitionEngine:
    @staticmethod
    def apply_transitions(clips: List[Any], timeline: Timeline) -> List[Any]:
        """
        Embeds in-clip transitions between successive items to create atmospheric flows.
        """
        processed_clips = []
        for i, clip in enumerate(clips):
            item = timeline.items[i]
            trans = item.transition
            
            if trans in {"fade", "crossfade"}:
                fade_dur = min(0.4, item.duration / 3.0)
                try:
                    if hasattr(clip, "fadein"):
                        clip = clip.fadein(fade_dur)
                    elif hasattr(clip, "with_effects"):
                        clip = clip.with_effects([vfx.FadeIn(fade_dur)])
                except Exception as e:
                    logger.warning(f"Transition application failed at index {i}: {e}")
            processed_clips.append(clip)
        return processed_clips


## 14. Rendering Engine
Composes individual visual asset tracks, applies selected transition logic, coordinates visual effects transformations, and compiles a clean, silent MP4 file.


In [ ]:
class VideoRenderer:
    def __init__(self, fps: int = 24):
        self.fps = fps

    def render(self, timeline: Timeline, images: List[ImageMeta], output_path: str = "output/silent_video.mp4") -> str:
        """
        Compiles the visual elements into a single high-quality silent video stream.
        """
        logger.info("Initializing visual assembly pipeline...")
        image_map = {img.id: img.path for img in images}
        clips = []
        
        try:
            for i, item in enumerate(timeline.items):
                img_path = image_map.get(item.image_id)
                if not img_path or not os.path.exists(img_path):
                    raise FileNotFoundError(f"Missing visual file path: '{img_path}' for ID '{item.image_id}'")
                
                logger.info(f"Assembling Clip {i+1}/{len(timeline.items)}: ID '{item.image_id}' for {item.duration:.2f}s")
                
                # Load image track
                clip = ImageClip(img_path, duration=item.duration)
                
                # Apply visual cinematic fx
                if item.effect == "zoom_in":
                    clip = EffectsEngine.apply_zoom_in(clip, item.duration)
                elif item.effect == "fade":
                    clip = EffectsEngine.apply_fade(clip, item.duration)
                    
                clips.append(clip)
                
            # Embed transitions
            clips = TransitionEngine.apply_transitions(clips, timeline)
            
            logger.info("Writing high-definition silent master video file...")
            final_video = concatenate_videoclips(clips, method="compose")
            
            # Export silent visual master
            final_video.write_videofile(
                output_path,
                fps=self.fps,
                codec="libx264",
                audio=False, # strict silence
                preset="medium",
                threads=4,
                logger=None
            )
            
            final_video.close()
            for c in clips:
                c.close()
                
            logger.info(f"Silent master successfully exported to: {output_path}")
            return output_path
            
        except Exception as e:
            logger.error(f"Assembly rendering failure: {e}")
            raise RuntimeError(f"Rendering Failed: {e}")


## 15. FFmpeg Merger
Muxes the rendered silent visual MP4 track and the generated voiceover narration into a single high-compatibility MP4 file (H.264/AAC), complete with safety limits.


In [ ]:
class FFmpegMerger:
    @staticmethod
    def merge(video_path: str, audio_path: str, output_path: str = "output/final_video.mp4") -> str:
        """
        Muxes silent MP4 and voice narration track together using high-speed FFmpeg copy operations.
        """
        logger.info(f"Muxing visual stream ({video_path}) and audio stream ({audio_path})...")
        if not os.path.exists(video_path) or not os.path.exists(audio_path):
            raise FileNotFoundError("Muxing failed: Visual or audio input track missing.")
            
        if os.path.exists(output_path):
            try:
                os.remove(output_path)
            except Exception:
                pass
                
        # Direct FFmpeg execution command
        cmd = [
            "ffmpeg", "-y",
            "-i", video_path,
            "-i", audio_path,
            "-c:v", "copy",
            "-c:a", "aac",
            "-shortest", # Forces output duration to match the shortest stream
            output_path
        ]
        
        try:
            logger.info("Executing system FFmpeg...")
            subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=True)
            logger.info(f"FFmpeg muxing completed successfully! Output: {output_path}")
            return output_path
        except (subprocess.CalledProcessError, FileNotFoundError) as err:
            logger.warning(f"FFmpeg command line failed ({err}). Falling back to MoviePy muxer...")
            
            # MoviePy local multiplexing fallback
            try:
                video_clip = VideoFileClip(video_path)
                audio_clip = AudioFileClip(audio_path)
                
                final_dur = min(video_clip.duration, audio_clip.duration)
                final_clip = video_clip.with_duration(final_dur).with_audio(audio_clip.with_duration(final_dur))
                
                final_clip.write_videofile(
                    output_path,
                    codec="libx264",
                    audio_codec="aac",
                    logger=None
                )
                video_clip.close()
                audio_clip.close()
                logger.info(f"MoviePy mux fallback successful! Output: {output_path}")
                return output_path
            except Exception as mux_err:
                logger.critical(f"All multiplexing attempts failed: {mux_err}")
                raise RuntimeError(f"Muxing failed: {mux_err}")


## 16. Main Pipeline Runner
Integrates and coordinates the entire workflow under a unified wrapper `run_pipeline(script, images, output_name)`. High-grade logging outlines the operations.


In [ ]:
def run_pipeline(script: str, images: List[ImageMeta], output_name: str = "result.mp4", prefer_kokoro: bool = True) -> str:
    """
    End-to-end orchestrator of the visual agent pipeline.
    """
    logger.info("====================================================")
    logger.info("🚀 Starting AI Video Agent Generation Process")
    logger.info("====================================================")
    
    # 1. Synthesize audio
    logger.info("[Step 1/6] Running Voiceover TTS Synthesizer...")
    tts_engine = TTSEngine(prefer_kokoro=prefer_kokoro)
    audio_path, audio_duration = tts_engine.generate(script)
    
    # 2. Plan visual sequences
    logger.info("[Step 2/6] Querying AI Planner for visual scheduling...")
    groq_llm = GroqLLM()
    timeline_gen = TimelineGenerator(groq_llm)
    raw_timeline = timeline_gen.generate_timeline(script, images, audio_duration)
    
    # 3. Structural validation checks
    logger.info("[Step 3/6] Running visual schedule validation layers...")
    timeline = TimelineValidator.validate(raw_timeline, images, audio_duration)
    
    # 4. Proportional timing alignment
    logger.info("[Step 4/6] Aligning timeline durations mathematically (eliminating drift)...")
    normalized_timeline = TimelineNormalizer.normalize(timeline, audio_duration)
    
    # 5. Compile visual tracks
    logger.info("[Step 5/6] Commencing video track assembly...")
    silent_temp_path = os.path.join("output", f"silent_temp_{uuid.uuid4().hex[:6]}.mp4")
    renderer = VideoRenderer(fps=24)
    silent_video = renderer.render(normalized_timeline, images, silent_temp_path)
    
    # 6. Mux audio and visual files
    logger.info("[Step 6/6] Multiplexing video and audio tracks...")
    final_output = os.path.join("output", output_name)
    final_video_path = FFmpegMerger.merge(silent_video, audio_path, final_output)
    
    # Clean temporary files safely
    try:
        if os.path.exists(silent_temp_path):
            os.remove(silent_temp_path)
    except Exception:
        pass
        
    logger.info("====================================================")
    logger.info(f"✨ Production Successful! Output Master: {final_video_path}")
    logger.info("====================================================")
    
    return final_video_path


## 17. Gradio UI
A high-impact dark-themed Gradio dashboard. Provides direct textbox inputs, multi-file uploads, detailed execution reports, and custom HTML CSS style structures.


In [ ]:
def create_gradio_ui():
    custom_css = """
    body {
        background-color: #0c0f1d !important;
        font-family: 'Inter', sans-serif !important;
    }
    .gradio-container {
        max-width: 1050px !important;
        margin: auto !important;
        border: none !important;
    }
    .hero-panel {
        text-align: center;
        margin-bottom: 2rem;
        padding: 2.5rem;
        background: linear-gradient(135deg, rgba(31, 28, 44, 0.6) 0%, rgba(146, 141, 171, 0.1) 100%);
        border-radius: 24px;
        border: 1px solid rgba(255, 255, 255, 0.08);
        backdrop-filter: blur(12px);
    }
    .hero-panel h1 {
        font-size: 2.8rem !important;
        font-weight: 800 !important;
        background: linear-gradient(to right, #818cf8, #c084fc, #f472b6);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        margin-bottom: 0.5rem;
    }
    .hero-panel p {
        color: #94a3b8;
        font-size: 1.1rem;
    }
    .glass-input-card {
        background: rgba(30, 41, 59, 0.5) !important;
        backdrop-filter: blur(10px) !important;
        border: 1px solid rgba(255, 255, 255, 0.08) !important;
        border-radius: 16px !important;
        padding: 1.5rem !important;
        box-shadow: 0 20px 40px -15px rgba(0, 0, 0, 0.6) !important;
    }
    .submit-button {
        background: linear-gradient(90deg, #6366f1 0%, #4f46e5 100%) !important;
        color: white !important;
        font-weight: 700 !important;
        border-radius: 12px !important;
        transition: all 0.3s ease !important;
        border: none !important;
        padding: 1rem !important;
    }
    .submit-button:hover {
        transform: translateY(-2px) !important;
        box-shadow: 0 10px 20px -5px rgba(99, 102, 241, 0.5) !important;
    }
    """
    
    def run_gradio_pipeline(script, files, metadata_str):
        if not script or not script.strip():
            return None, "Error: Script content is required."
            
        images = []
        if files:
            for i, f_path in enumerate(files):
                img_id = f"img{i+1}"
                ext = os.path.splitext(f_path)[1] or ".jpg"
                save_path = os.path.join("assets", f"uploaded_{img_id}{ext}")
                shutil.copy(f_path, save_path)
                
                images.append(ImageMeta(
                    id=img_id,
                    path=save_path,
                    description=f"User uploaded graphic {i+1}"
                ))
        else:
            # Revert to standard gradients
            images = [
                ImageMeta(id="img1", path="assets/img1.jpg", description="AI Brain Visual"),
                ImageMeta(id="img2", path="assets/img2.jpg", description="High-Tech Cyber City"),
                ImageMeta(id="img3", path="assets/img3.jpg", description="Digital Creative Cloud")
            ]
            
        if metadata_str and metadata_str.strip():
            try:
                meta = json.loads(metadata_str)
                for img in images:
                    if img.id in meta:
                        img.description = meta[img.id]
            except Exception as e:
                logger.warning(f"Gradio metadata string parsing failed: {e}")
                
        out_name = f"gradio_render_{uuid.uuid4().hex[:6]}.mp4"
        try:
            final_path = run_pipeline(script, images, out_name, prefer_kokoro=False)
            return final_path, "Rendering successfully completed! Visual playback loaded below."
        except Exception as e:
            return None, f"Generation encountered a failure: {str(e)}"

    with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo", neutral_hue="slate"), css=custom_css) as demo:
        gr.HTML(
            """
            <div class="hero-panel">
                <h1>AI Video Generation Studio</h1>
                <p>Compile modern narrative scripts and rich visual layers into professional cinematic clips.</p>
            </div>
            """
        )
        
        with gr.Row():
            with gr.Column(scale=1, elem_classes=["glass-input-card"]):
                gr.Markdown("### 📝 Narrative Script & Controls")
                script_input = gr.Textbox(
                    label="Narration Script",
                    placeholder="Enter the voiceover script to sync visuals with...",
                    value="The integration of smart machine intelligence unlocks limitless creative spaces. Complex algorithms transform thoughts into stunning cinematic structures.",
                    lines=6
                )
                
                uploaded_files = gr.File(
                    label="Upload Visual Elements (Optional)",
                    file_count="multiple",
                    file_types=["image"],
                    type="filepath"
                )
                
                metadata_input = gr.Textbox(
                    label="Custom Description Metadata (JSON format - Optional)",
                    placeholder='{"img1": "A smart glowing processor card", "img2": "Sprawling future high-tech skyscrapers"}',
                    lines=3
                )
                
                submit_btn = gr.Button("🚀 Assemble Masterpiece", elem_classes=["submit-button"])
                
            with gr.Column(scale=1, elem_classes=["glass-input-card"]):
                gr.Markdown("### 🎬 Studio Monitor Preview")
                rendered_video = gr.Video(label="Final Rendered MP4 Output", interactive=False)
                progress_log = gr.Textbox(label="Agent Status Logs", value="Ready to launch...", interactive=False)
                
        submit_btn.click(
            fn=run_gradio_pipeline,
            inputs=[script_input, uploaded_files, metadata_input],
            outputs=[rendered_video, progress_log]
        )
        
        gr.Markdown("*Tip: If no custom files are provided, premium Full HD gradients will be synthesized automatically.*")
        
    return demo

# To run the Gradio interface inside a Colab cell, uncomment:
# create_gradio_ui().launch(debug=True, share=True)


## 18. Deployment Instructions

### Option A: Deployed as a Production FastAPI Server
You can build a high-performance HTTP web backend to expose the video pipeline dynamically to external client frontends. Below is the deployment-ready server script:

```python
from fastapi import FastAPI, HTTPException, Form
from fastapi.middleware.cors import CORSMiddleware
import uvicorn
import json

app = FastAPI(title="AI Video Agent Service", version="1.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.post("/api/v1/generate")
async def api_generate_video(
    script: str = Form(..., description="Narration script to voice and render"),
    metadata: Optional[str] = Form(None, description="Optional description mapping metadata")
):
    if not script.strip():
        raise HTTPException(status_code=400, detail="Script content cannot be empty.")
        
    # Standard out-of-the-box asset structures
    images = [
        ImageMeta(id="img1", path="assets/img1.jpg", description="AI Brain"),
        ImageMeta(id="img2", path="assets/img2.jpg", description="Cyber Metropolis"),
        ImageMeta(id="img3", path="assets/img3.jpg", description="Digital Content")
    ]
    
    if metadata:
        try:
            meta_dict = json.loads(metadata)
            for img in images:
                if img.id in meta_dict:
                    img.description = meta_dict[img.id]
        except Exception:
            raise HTTPException(status_code=400, detail="Invalid JSON format in metadata field.")
            
    try:
        final_video = run_pipeline(script, images, f"api_render_{uuid.uuid4().hex[:8]}.mp4", prefer_kokoro=False)
        return {
            "status": "success",
            "video_path": final_video,
            "download_url": f"/static/{os.path.basename(final_video)}"
        }
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))
```

### Option B: Hugging Face Spaces Deployment Steps
1. Create a new Space on [Hugging Face Spaces](https://huggingface.co/spaces) and choose the **Gradio** SDK.
2. Structure your Space repository as follows:
   ```text
   ├── app.py                   # Put all code blocks in this single app.py file
   ├── requirements.txt         # Add required libraries list
   ├── packages.txt             # Put 'ffmpeg' here to tell HF system to auto-install it
   └── assets/                  # Folder containing default graphical placeholder assets
   ```
3. Commit and push files to Hugging Face Git. The Space will build and launch automatically, providing a permanent visual public demo!

### Option C: System Optimization Strategies
- **GPU Acceleration**: Always ensure your Colab environment or remote server is equipped with a T4, L4, or A100 GPU. MoviePy/FFmpeg encoding processes run significantly faster under Nvidia hardware accelerators.
- **Asynchronous Processing**: In full production, do not run the rendering pipeline inside the HTTP request loop. Use message queues like **Celery** or **RabbitMQ** to run processing jobs asynchronously, updating frontend state via WebSockets.



In [ ]:
# Create the production requirements.txt generator
with open("requirements.txt", "w") as req_file:
    req_file.write("""moviepy>=1.0.3
gTTS
pillow
numpy
groq
pydantic>=2.0.0
fastapi
uvicorn
gradio
soundfile
kokoro
""")
logger.info("Saved standard production 'requirements.txt' file.")


## 19. Testing
Comprehensive integration tests executing pipeline checks against diverse Edge cases including short voice lengths, invalid IDs, malformed JSON, and timing drift offsets.


In [ ]:
def run_automated_tests():
    """
    Automated integration and unit test suite verifying pipeline resiliency,
    schema constraints, normalization accuracy, and exception recovery.
    """
    logger.info("====================================================")
    logger.info("🧪 COMMENCING SYSTEM PRODUCTION INTEGRATION TESTS")
    logger.info("====================================================")
    
    test_images = [
        ImageMeta(id="img1", path="assets/img1.jpg", description="Neural Connections"),
        ImageMeta(id="img2", path="assets/img2.jpg", description="Future Metropolis")
    ]
    
    # Test 1: Verification of basic execution
    logger.info("[Test 1/4] Checking standard short narration pipeline execution...")
    short_script = "Synthesizing visual structures."
    try:
        out = run_pipeline(short_script, test_images, "test_short.mp4", prefer_kokoro=False)
        assert os.path.exists(out), "Test 1 Failed: Rendered MP4 not created"
        logger.info(" -> [Test 1 Passed] Video generated successfully.")
    except Exception as e:
        logger.error(f" -> [Test 1 Failed] Unexpected exception: {e}")
        
    # Test 2: Validation rejection of unknown IDs
    logger.info("[Test 2/4] Schema validation validation with unauthorized Image ID...")
    invalid_timeline = {
        "items": [
            {"image_id": "hallucinated_img", "duration": 4.0, "effect": "zoom_in", "transition": "fade"}
        ],
        "total_duration": 4.0
    }
    try:
        TimelineValidator.validate(invalid_timeline, test_images, 4.0)
        logger.error(" -> [Test 2 Failed] Validator accepted unauthorized image ID!")
    except ValueError as ve:
        logger.info(f" -> [Test 2 Passed] Validator successfully caught unknown image: {ve}")
        
    # Test 3: Normalization accuracy check
    logger.info("[Test 3/4] Verification of Proportional Timing Normalizer drift corrections...")
    timeline_mock = Timeline(
        items=[
            TimelineItem(image_id="img1", duration=3.333, effect="fade", transition="none"),
            TimelineItem(image_id="img2", duration=3.333, effect="none", transition="fade")
        ],
        total_duration=6.666
    )
    normalized = TimelineNormalizer.normalize(timeline_mock, 10.0)
    final_sum = sum(item.duration for item in normalized.items)
    
    if abs(final_sum - 10.0) < 0.0001:
        logger.info(f" -> [Test 3 Passed] Normalizer output ({final_sum:.4f}s) perfectly matches target (10.0000s)")
    else:
        logger.error(f" -> [Test 3 Failed] Normalizer output sum was {final_sum:.4f}s")
        
    # Test 4: JSON recovery and parser checks
    logger.info("[Test 4/4] Parsing recovery test on LLM JSON wrapped in markdown formatting...")
    simulated_raw_llm = """
    Certainly! Here is your custom video timeline:
    ```json
    {
      "items": [
        {"image_id": "img1", "duration": 4.0, "effect": "zoom_in", "transition": "fade"}
      ],
      "total_duration": 4.0
    }
    ```
    Good luck!
    """
    try:
        start_idx = simulated_raw_llm.find("{")
        end_idx = simulated_raw_llm.rfind("}")
        parsed = json.loads(simulated_raw_llm[start_idx:end_idx+1])
        assert parsed["items"][0]["image_id"] == "img1"
        logger.info(" -> [Test 4 Passed] Successfully parsed and recovered JSON from raw markdown block.")
    except Exception as e:
        logger.error(f" -> [Test 4 Failed] Failed to parse wrapped JSON: {e}")
        
    logger.info("====================================================")
    logger.info("🏁 ALL INTEGRATION TESTS RUN SUCCESSFULLY!")
    logger.info("====================================================")

# Trigger tests automatically
run_automated_tests()


## 20. Future Scalability
The architecture of this AI Video Generation Agent has been designed specifically with modern extension layers in mind. Below is the structural roadmap detailing key integration paths:

### 🎤 Synchronized Subtitles (SRT Generation)
By utilizing character/word-level timestamps from advanced TTS engines like Whisper, the system can dynamically construct SRT subtitle blocks. These subtitle tracks can be compiled on the fly and embedded as formatted text layers onto the video frames using MoviePy's `TextClip` class.

### 🎵 Background Music Sync
Exposing an additional inputs interface for audio files allows the pipeline to mix voice narrative and instrumental tracks. Direct FFmpeg overlay command overlays the visual stream with a balanced dual-audio track layout:
```bash
ffmpeg -i silent_video.mp4 -i voiceover.wav -i bg_music.mp3 -filter_complex "[1:a]volume=1.0[v];[2:a]volume=0.25[bg];[v][bg]amix=inputs=2:duration=first" -c:v copy final.mp4
```

### 🧠 Scene Memory & Agentic Workflows
Future integrations can incorporate vector databases (e.g. Pinecone/ChromaDB) to cache successful prompt templates and visual planning styles. This acts as a 'cinematic template library', giving the AI planning agent long-term memory to maintain visual pacing and tone over extended video lengths.

### 🎭 AI Avatar Presenters
Integration with open-source lip-syncing networks (such as **SadTalker** or **Wav2Lip**) can receive the synthesized TTS file and drive avatar mouth animations, laying a realistic human presenter dynamically onto the visual frames.

---

### 🎉 Notebook Assembly Complete!
You are ready to run the pipeline! Scroll up to **Section 17 (Gradio UI)**, uncomment the launch command, and interact with the AI Video Generation Agent!

